In [28]:
import os
import jax
jax.config.update("jax_enable_x64", True)
from pyscf import gto, scf, cc

a = 1.20577 # bond length in a cluster
d = 100 # distance between each cluster
unit = 'A' # unit of length
na = 2 # size of a cluster (monomer)
nc_list = [1] # set as integer multiple of monomers
spin = 2 # spin per monomer
frozen = 0 # frozen orbital per monomer
elmt = 'N'
basis = 'ccpvtz'
for nc in nc_list:
    atoms = ""
    for n in range(nc*na):
        shift = ((n - n % na) // na) * (d-a)
        atoms += f"{elmt} {n*a+shift:.5f} 0.00000 0.00000 \n"

    mol = gto.M(atom=atoms,basis=basis,spin=spin*nc,unit=unit,verbose=4)
    mol.build()

    mf = scf.UHF(mol)
    mf.kernel()

    stable = False
    for i in range(10):
        print(f'mf stability test {i+1}')
        if not stable:
            mo_i, _, stable,_ = mf.stability(return_status=True)
            dm = mf.make_rdm1(mo_i,mf.mo_occ)
            mf.kernel(dm0=dm)
        elif stable:
            print(f'mf energy: {mf.e_tot}, stability {stable}')
            break

    mycc = cc.CCSD(mf,frozen=2*nc)
    mycc.kernel()

System: uname_result(system='Linux', node='sharmagroup-rn', release='6.17.0-29-generic', version='#29~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Mon May 11 10:30:58 UTC 2', machine='x86_64')  Threads 16
Python 3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:20:58) [GCC 14.3.0]
numpy 2.4.4  scipy 1.17.1  h5py 3.16.0
Date: Wed May 27 13:32:35 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] OLD_PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:
[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:/home/sharmagroup/sharmagroup/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 2
[INPUT] num. electrons = 14
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 2
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = A
[INPUT] Symbol           X                Y                Z     

In [ ]:
# def prep_integral(
#     mf_or_cc: Union[scf.rhf.RHF, scf.uhf.UHF, CCSD, UCCSD],
#     basis_coeff: Optional[np.ndarray] = None,
#     norb_frozen: int = 0,
#     chol_cut: float = 1e-5,
#     save2disk = True,
#     amp_file = "amplitudes.npz",
#     chol_file = "FCIDUMP_chol"
# ):

#     print("\nPreparing AFQMC calculation")

#     if isinstance(mf_or_cc, (CCSD, UCCSD)):
#         mf = mf_or_cc._scf
#         cc = mf_or_cc
#         if cc.frozen is not None:
#             norb_frozen = cc.frozen
#         if isinstance(cc, UCCSD):
#             # spin_type = 'unrestricted'
#             t1a = np.array(cc.t1[0])
#             t1b = np.array(cc.t1[1])
#             t2aa, t2ab, t2bb = cc.t2
#             t2aa = (t2aa - t2aa.transpose(0, 1, 3, 2)) / 2
#             t2bb = (t2bb - t2bb.transpose(0, 1, 3, 2)) / 2
#             t2aa = t2aa.transpose(0, 2, 1, 3)
#             t2bb = t2bb.transpose(0, 2, 1, 3)
#             t2ab = t2ab.transpose(0, 2, 1, 3)
#             np.savez(
#                 amp_file,
#                 t1a=t1a,
#                 t1b=t1b,
#                 t2aa=t2aa,
#                 t2ab=t2ab,
#                 t2bb=t2bb,
#             )
#         elif isinstance(cc, CCSD):
#             # spin_type = 'restricted'
#             t2 = cc.t2
#             t2 = t2.transpose(0, 2, 1, 3)
#             t1 = np.array(cc.t1)
#             np.savez(amp_file, t1=t1, t2=t2)
#     else:
#         mf = mf_or_cc

#     if isinstance(mf, scf.rhf.RHF):
#         spin_type = 'restricted'
#     elif isinstance(mf, scf.uhf.UHF):
#         spin_type = 'unrestricted'

#     mol = mf.mol
#     nao = mf.mol.nao

#     if basis_coeff is None:
#         basis_coeff = mf.mo_coeff

#     print("Calculating Cholesky integrals")
    
#     if getattr(mf, "with_df", None) is not None:
#         print('Find Density Fit Teonsers in MF object')
#         print('Integrals will be built by DF Tensors')
#         useDF = True
#     else:
#         useDF = False

#     if spin_type == 'restricted':

#         nbasis = nao - norb_frozen
#         nocc = int(np.count_nonzero(mf.mo_occ))
#         nelec = [nocc - norb_frozen, nocc - norb_frozen]
#         h1e, enuc = h1e_ras(mf, basis_coeff, nbasis, norb_frozen, useDF)
#         chol_ao = cholesky.cholesky_by_mol(mol, max_error=chol_cut, cmax=10)
#         chol_ao = jnp.array(chol_ao.reshape((-1, nao, nao)))
#         chol = cholesky.cderi2mo_gpu(chol_ao, basis_coeff)
#         chol = cholesky.unpack_symmetric(chol, nao)
#         chol = chol[:, norb_frozen:, norb_frozen:]

#         v0 = 0.5 * oe.contract("gpr,gqr->pq", chol, chol, backend="jax")
#         h1e_mod = h1e - v0
#         chol = chol.reshape((chol.shape[0], -1))
            
#     elif spin_type == 'unrestricted':

#         ncore = np.array([norb_frozen, norb_frozen], dtype = np.int32)
#         nocc = np.array([np.count_nonzero(mf.mo_occ[0]),
#                          np.count_nonzero(mf.mo_occ[1])],
#                          dtype = np.int32)
#         nelec = nocc - norb_frozen
#         ncas = nao - ncore
#         nbasis = ncas[0]
#         h1e, enuc = h1e_uas(mf, basis_coeff, ncas, ncore, useDF)

#         chol_ao = cholesky.cholesky_by_mol(mol, max_error=chol_cut, cmax=10)
#         chol_ao = jnp.array(chol_ao.reshape((-1, nao, nao)))
#         chol_a = cholesky.cderi2mo_gpu(chol_ao, basis_coeff[0])
#         chol_b = cholesky.cderi2mo_gpu(chol_ao, basis_coeff[1])
#         chol_a = cholesky.unpack_symmetric(chol_a, nao)
#         chol_b = cholesky.unpack_symmetric(chol_b, nao)
#         print(f"Alpha Cholesky shape: {chol_a.shape} ")
#         print(f" Beta Cholesky shape: {chol_b.shape} ")

#         chol_a = chol_a[:, ncore[0]:, ncore[0]:]
#         chol_b = chol_b[:, ncore[1]:, ncore[1]:]
#         v0_a = 0.5 * oe.contract("gpr,gqr->pq", chol_a, chol_a, backend="jax")
#         v0_b = 0.5 * oe.contract("gpr,gqr->pq", chol_b, chol_b, backend="jax")
#         h1e = jnp.array(h1e)
#         h1e_mod = jnp.array(h1e - jnp.array([v0_a,v0_b]))
#         chol = jnp.array([chol_a.reshape(chol_a.shape[0], -1), chol_b.reshape(chol_b.shape[0], -1)])

#     print("Finished calculating Cholesky integrals")
#     print("Size of the correlation space:")
#     print(f"Number of electrons:        {nelec}")
#     print(f"Number of basis functions:  {nbasis}")
#     print(f"Number of Cholesky vectors: {chol.shape[-2]}")

#     if save2disk:
#         integral.write_integral(
#                 enuc=enuc,
#                 hcore=h1e,
#                 hcore_mod=h1e_mod,
#                 chol=chol,
#                 nelec=sum(nelec),
#                 nmo=nbasis,
#                 ms=mol.spin,
#                 spin_type=spin_type,
#                 filename=chol_file,
#             )

In [29]:
from afqmc import integral
integral.prep_integral(mycc)


Preparing AFQMC calculation
Calculating Cholesky integrals
Alpha Cholesky shape: (358, 60, 60) 
 Beta Cholesky shape: (358, 60, 60) 
Finished calculating Cholesky integrals
Size of the correlation space:
Number of electrons:        [6 4]
Number of basis functions:  58
Number of Cholesky vectors: 358


In [53]:
options = {
    'eql_time': 10,
    'n_blocks': 1000,
    'n_walkers': 1,
    'max_error': 0.0,
    'mix_precision': False,
    'seed': 17,
    'walker_type': 'uhf',
    'trial': 'upt2ccsd_ad',
    }

# import pickle
# with open('options.bin', 'wb') as f:
#     pickle.dump(options, f)

In [30]:
import os
os.environ.setdefault("XLA_PYTHON_CLIENT_ALLOCATOR", "platform")

import h5py
import pickle

import numpy as np

import opt_einsum as oe
import jax.numpy as jnp
from jax import scipy as jsp
from jax import jit, lax, random, vmap

from afqmc import hamiltonian, cholesky
from afqmc import propagation, sampling, fp_sampling
from afqmc.wavefunctions import wavefunctions_restricted
from afqmc.wavefunctions import wavefunctions_unrestricted

In [58]:
def init_afqmc(options=None,
               option_file="options.bin",
               amp_file="amplitudes.npz",
               chol_file="FCIDUMP_chol"):
    
    if options is None:
        try:
            with open(option_file, "rb") as f:
                options = pickle.load(f)
        except:
            options = {}

    options["dt"] = options.get("dt", 0.005)
    options["n_exp_terms"] = options.get("n_exp_terms",6)
    options["n_walkers"] = options.get("n_walkers", 50)
    options["n_prop_steps"] = options.get("n_prop_steps", 50)
    options["n_blocks"] = options.get("n_blocks", 500)
    options["seed"] = options.get("seed", np.random.randint(1, int(1e6)))
    options["eql_time"] = options.get("eql_time", 20)
    options["walker_type"] = options.get("walker_type", "rhf")
    options["trial"] = options.get("trial", None)
    options["n_batch"] = options.get("n_batch", 1)
    options["max_error"] = options.get("max_error", 0.0)
    options["nchol_chunk"] = options.get("nchol_chunk", 100)
    options["max_memory"] = options.get("max_memory", 2000) # MB
    options["mix_precision"] = options.get("mix_precision", True)
    options["free_projection"] = options.get("free_projection", False)

    print("\nLoad system from Integral File")

    with h5py.File(chol_file, "r") as fh5:
        [nelec, norb, ms] = fh5["header"]
        spin_type = fh5["spin_type"][()]
        h0 = jnp.array(fh5.get("energy_core"))
        h1 = jnp.array(fh5.get("hcore"))
        chol = jnp.array(fh5.get("chol"))
        h1_mod = jnp.array(fh5.get("hcore_mod"))
    
    if isinstance(spin_type, bytes):
        spin_type = spin_type.decode()

    assert spin_type in ["restricted", "unrestricted"]

    # print(f"AFQMC Object Spin type: {spin_type}")

    if spin_type == 'restricted':
        h1 = jnp.array(h1).reshape(norb, norb)
        h1_mod = jnp.array(h1_mod).reshape(norb, norb)
        chol = jnp.array(chol).reshape(-1, norb, norb)

    elif spin_type == 'unrestricted':
        h1 = jnp.array(h1).reshape(2, norb, norb)
        h1_mod = jnp.array(h1_mod).reshape(2, norb, norb)
        chol = jnp.array(chol).reshape(2, -1, norb, norb)

    assert type(ms) is np.int64
    assert type(nelec) is np.int64
    assert type(norb) is np.int64

    ms, nelec, norb = int(ms), int(nelec), int(norb)
    nelec_sp = ((nelec + abs(ms)) // 2, (nelec - abs(ms)) // 2)

    ham = hamiltonian.hamiltonian(norb)
    ham_data = {}
    ham_data["h0"] = h0

    if spin_type == 'restricted':
        ham_data["h1"] = jnp.array([h1, h1])
        ham_data["h1_mod"] = jnp.array(h1_mod)
        nchol = chol.shape[0]
        ham_data["chol"] = jnp.array(chol.reshape(chol.shape[0], -1))
    elif spin_type == 'unrestricted':
        ham_data["h1"] = jnp.array(h1)
        ham_data["h1_mod"] = jnp.array(h1_mod)
        nchol = chol[0].shape[0]
        ham_data["chol"] = jnp.array([chol[0].reshape(chol[0].shape[0], -1),
                                      chol[1].reshape(chol[1].shape[0], -1)])

    options["nchol_chunk"] = cholesky.chunk_chol(
        chol, options["nchol_chunk"], options["max_memory"]/options["n_walkers"])

    wave_data = {}
    mo_coeff = jnp.array([np.eye(norb),np.eye(norb)])

    print(spin_type)

    if spin_type == "restricted":
        if options["trial"] == "rhf":
            trial = wavefunctions_restricted.rhf(norb, nelec_sp, 
                                                 n_batch=options["n_batch"],
                                                 nchol_chunk=options["nchol_chunk"],
                                                 )
            wave_data["mo_coeff"] = mo_coeff[0][:, : nelec_sp[0]]

        elif "cisd" in options["trial"]:
            try:
                amplitudes = np.load(amp_file)
                t1 = jnp.array(amplitudes["t1"])
                t2 = jnp.array(amplitudes["t2"])
                ci2 = t2 + jnp.einsum("ia,jb->iajb", t1, t1)
                trial_wave_data = {"ci1": t1, "ci2": ci2, 
                                "mo_coeff": mo_coeff[0][:, : nelec_sp[0]]}
                wave_data.update(trial_wave_data)
                trial = wavefunctions_restricted.cisd(norb, nelec_sp, 
                                                      n_batch=options["n_batch"]
                                                      )
                if "/" in options["trial"]:
                    guide_wave = wavefunctions_restricted.cisd(norb, nelec_sp, n_batch=options["n_batch"])
                    trial_wave = wavefunctions_restricted.rhf(norb, nelec_sp, n_batch=options["n_batch"])
                    trial = wavefunctions_restricted.mixed(guide_wave, trial_wave)
            except:
                raise ValueError("Trial specified as cisd, but amplitudes.npz not found.")

        elif options["trial"] == "cid":
            try:
                amplitudes = np.load(amp_file)
                t2 = jnp.array(amplitudes["t2"])
                trial_wave_data = {"ci2": t2, "mo_coeff": mo_coeff[0][:, : nelec_sp[0]]}
                wave_data.update(trial_wave_data)
                trial = wavefunctions_restricted.cid(norb, nelec_sp, n_batch=options["n_batch"])
            except:
                raise ValueError("Trial specified as cisd, but amplitudes.npz not found.")
            
        elif "ptccsd" in options["trial"]:
            amplitudes = np.load(amp_file)
            t1 = jnp.array(amplitudes["t1"])
            t2 = jnp.array(amplitudes["t2"])
            trial_wave_data = {"t1": t1, "t2": t2}
            wave_data.update(trial_wave_data)
            wave_data["mo_coeff"] = mo_coeff[0][:,:nelec_sp[0]]
            trial = wavefunctions_restricted.ptccsd(norb, nelec_sp, n_batch=options["n_batch"])
            if "ad" in options["trial"]:
                trial = wavefunctions_restricted.ptccsd_ad(norb, nelec_sp, n_batch=options["n_batch"])
        
        elif "ptccd" in options["trial"]:
            amplitudes = np.load(amp_file)
            t2 = jnp.array(amplitudes["t2"])
            trial_wave_data = {"t2": t2}
            wave_data.update(trial_wave_data)
            wave_data["mo_coeff"] = mo_coeff[0][:,:nelec_sp[0]]
            trial = wavefunctions_restricted.ptccd(norb, nelec_sp, n_batch=options["n_batch"])

        elif "pt2ccsd"in options["trial"]:
            trial = wavefunctions_restricted.pt2ccsd(norb, nelec_sp, 
                                                     n_batch=options["n_batch"],
                                                     nchol_chunk=options["nchol_chunk"], 
                                                     mix_precision=options["mix_precision"],
                                                     )
            nocc = nelec_sp[0]
            amplitudes = np.load(amp_file)
            t1 = jnp.array(amplitudes["t1"])
            t2 = jnp.array(amplitudes["t2"])
            trial_wave_data = {"t1": t1, "t2": t2}
            wave_data.update(trial_wave_data)
            mo_t = trial.thouless_trans(t1)[:,:nocc]
            wave_data['mo_t'] = mo_t
            wave_data["mo_coeff"] = mo_coeff[0][:,:nelec_sp[0]]
            if "ad" in options["trial"]:
                trial_ad = wavefunctions_restricted.pt2ccsd_ad(norb, nelec_sp, 
                                                            n_batch=options["n_batch"])
                rot_t2 = jnp.einsum('il,jk,lakb->iajb',
                                mo_t[:nocc,:nocc].T,mo_t[:nocc,:nocc].T,t2)
                wave_data['rot_t2'] = rot_t2

        elif "stoccsd" in options["trial"]:
            if "2" in options["trial"]:
                trial = wavefunctions_restricted.stoccsd2(
                    norb,
                    nelec_sp,
                    n_batch = options["n_batch"],
                    nslater = options['nslater']
                    )
                    
                sampler = sampling.sampler_stoccsd2(
                    n_prop_steps = options["n_prop_steps"],
                    n_blocks = options["n_blocks"],
                    n_chol = nchol,
                    )
            else:
                trial = wavefunctions_restricted.stoccsd(
                    norb,
                    nelec_sp,
                    n_batch = options["n_batch"],
                    nslater = options['nslater']
                    )
                    
                sampler = sampling.sampler_stoccsd(
                    n_prop_steps = options["n_prop_steps"],
                    n_blocks = options["n_blocks"],
                    n_chol = nchol,
                    )
            
            nocc = nelec_sp[0]
            amplitudes = np.load(amp_file)
            t1 = jnp.array(amplitudes["t1"])
            t2 = jnp.array(amplitudes["t2"])
            trial_wave_data = {"t1": t1, "t2": t2}
            wave_data.update(trial_wave_data)
            init_sd = jnp.eye(norb)[:,:nocc]
            mo_t = trial._thouless(init_sd, t1)
            wave_data['mo_t'] = mo_t
            wave_data['tau'] = trial.decompose_t2(t2)
            wave_data["mo_coeff"] = mo_coeff[0][:,:nocc]
    
    elif spin_type == "unrestricted":
        if options["trial"] == "uhf":
            trial = wavefunctions_unrestricted.uhf(norb, nelec_sp, 
                                                   n_batch=options["n_batch"])
            wave_data["mo_coeff"] = [
                mo_coeff[0][:, : nelec_sp[0]],
                mo_coeff[1][:, : nelec_sp[1]],
            ]

        elif options["trial"] == "ucisd":
            trial = wavefunctions_unrestricted.ucisd(
                    norb, nelec_sp, n_batch=options["n_batch"])
            nocc_a, nocc_b = trial.nelec[0], trial.nelec[1]
            try:
                amplitudes = np.load(amp_file)
                t1a = jnp.array(amplitudes["t1a"])
                t1b = jnp.array(amplitudes["t1b"])
                t2aa = jnp.array(amplitudes["t2aa"])
                t2ab = jnp.array(amplitudes["t2ab"])
                t2bb = jnp.array(amplitudes["t2bb"])
                ci2aa = t2aa + 2 * jnp.einsum("ia,jb->iajb", t1a, t1a)
                ci2ab = t2ab + jnp.einsum("ia,jb->iajb", t1a, t1b)
                ci2bb = t2bb + 2 * jnp.einsum("ia,jb->iajb", t1b, t1b)
                ci2aa = (ci2aa - ci2aa.transpose(0, 3, 2, 1)) / 2
                ci2bb = (ci2bb - ci2bb.transpose(0, 3, 2, 1)) / 2
                trial_wave_data = {
                    "ci1A": t1a,
                    "ci1B": t1b,
                    "ci2AA": ci2aa,
                    "ci2AB": ci2ab,
                    "ci2BB": ci2bb,
                    "mo_coeff": mo_coeff,
                }
                wave_data.update(trial_wave_data)
                mo = [mo_coeff[0][:,:nocc_a], mo_coeff[1][:,:nocc_b]]
                mo_t = trial._thouless(mo, [t1a, t1b])
                wave_data['mo_ta'] = mo_t[0]
                wave_data['mo_tb'] = mo_t[1]
                # wave_data['tau'] = trial.decompose_t2([t2aa,t2ab,t2bb])
            except:
                raise ValueError("Trial specified as ucisd, but amplitudes.npz not found.")

        elif "uptccsd" in options["trial"]:
            trial = wavefunctions_unrestricted.uptccsd(norb, nelec_sp, n_batch = options["n_batch"])
            noccA, noccB = trial.nelec[0], trial.nelec[1]
            wave_data["mo_coeff"] = [
                mo_coeff[0][:, : noccA],
                mo_coeff[1][:, : noccB],
            ]
            ham_data['h1_mod'] = h1_mod
            amplitudes = np.load(amp_file)
            t1a = jnp.array(amplitudes["t1a"])
            t1b = jnp.array(amplitudes["t1b"])
            t2aa = jnp.array(amplitudes["t2aa"])
            t2ab = jnp.array(amplitudes["t2ab"])
            t2bb = jnp.array(amplitudes["t2bb"])
            wave_data['t1a'] = t1a
            wave_data['t1b'] = t1b
            wave_data["t2aa"] = t2aa
            wave_data["t2bb"] = t2bb
            wave_data["t2ab"] = t2ab
            if "ad" in options["trial"]:
                trial_ad = wavefunctions_unrestricted.uptccsd_ad(
                    norb, nelec_sp, n_batch=options["n_batch"])
                mo_a_A = wave_data['mo_coeff'][0]
                mo_b_B = wave_data['mo_coeff'][1]
                wave_data["rot_t1A"] = mo_a_A[:noccA,:noccA].T @ t1a
                wave_data["rot_t2AA"] = jnp.einsum('ik,jl,kalb->iajb',
                    mo_a_A[:noccA,:noccA].T,mo_a_A[:noccA,:noccA].T,t2aa)
                wave_data["rot_t1B"] = mo_b_B[:noccB,:noccB].T @ t1b
                wave_data["rot_t2BB"] = jnp.einsum('ik,jl,kalb->iajb',
                    mo_b_B[:noccB,:noccB].T,mo_b_B[:noccB,:noccB].T,t2bb)
                wave_data["rot_t2AB"] = jnp.einsum('ik,jl,kalb->iajb',
                    mo_a_A[:noccA,:noccA].T,mo_b_B[:noccB,:noccB].T,t2ab)

        elif "upt2ccsd" in options["trial"]:
            trial = wavefunctions_unrestricted.upt2ccsd(
                norb, nelec_sp, 
                n_batch=options["n_batch"], 
                nchol_chunk=options["nchol_chunk"],
                mix_precision=options["mix_precision"],
                )
            noccA, noccB = trial.nelec[0], trial.nelec[1]
            wave_data["mo_coeff"] = [
                mo_coeff[0][:, : noccA],
                mo_coeff[1][:, : noccB],
            ]
            ham_data['h1_mod'] = h1_mod
            amplitudes = np.load(amp_file)
            t1a = jnp.array(amplitudes["t1a"])
            t1b = jnp.array(amplitudes["t1b"])
            t2aa = jnp.array(amplitudes["t2aa"])
            t2ab = jnp.array(amplitudes["t2ab"])
            t2bb = jnp.array(amplitudes["t2bb"])
            mo_ta = trial.thouless_trans(t1a)[:,:noccA]
            mo_tb = trial.thouless_trans(t1b)[:,:noccB]
            wave_data['mo_ta'] = mo_ta
            wave_data['mo_tb'] = mo_tb
            wave_data["t2aa"] = t2aa
            wave_data["t2bb"] = t2bb
            wave_data["t2ab"] = t2ab
            # wave_data['tau'] = trial.decompose_t2([t2aa,t2ab,t2bb])
            if "ad" in options["trial"]:
                trial_ad = wavefunctions_unrestricted.upt2ccsd_ad(
                    norb, nelec_sp, n_batch=options["n_batch"])
                wave_data["rot_t2aa"] = jnp.einsum('ik,jl,kalb->iajb',
                    mo_ta[:noccA,:noccA].T,mo_ta[:noccA,:noccA].T,t2aa)
                wave_data["rot_t2bb"] = jnp.einsum('ik,jl,kalb->iajb',
                    mo_tb[:noccB,:noccB].T,mo_tb[:noccB,:noccB].T,t2bb)
                wave_data["rot_t2ab"] = jnp.einsum('ik,jl,kalb->iajb',
                    mo_ta[:noccA,:noccA].T,mo_tb[:noccB,:noccB].T,t2ab)
            if "eff" in options["trial"]:
                trial = wavefunctions_unrestricted.upt2ccsd_eff(
                    norb, nelec_sp, n_batch=options["n_batch"])
                wave_data["rot_t2aa"] = jnp.einsum('ik,jl,kalb->iajb',
                    mo_ta[:noccA,:noccA].T,mo_ta[:noccA,:noccA].T,t2aa)
                wave_data["rot_t2bb"] = jnp.einsum('ik,jl,kalb->iajb',
                    mo_tb[:noccB,:noccB].T,mo_tb[:noccB,:noccB].T,t2bb)
                wave_data["rot_t2ab"] = jnp.einsum('ik,jl,kalb->iajb',
                    mo_ta[:noccA,:noccA].T,mo_tb[:noccB,:noccB].T,t2ab)

        elif options["trial"] == "ustoccsd2":
            trial = wavefunctions_unrestricted.ustoccsd2(
                norb,
                nelec_sp,
                n_batch = options["n_batch"],
                nslater = options['nslater']
                )
            nocc_a, nocc_b = nelec_sp
            amplitudes = np.load(amp_file)
            t1a = jnp.array(amplitudes["t1a"])
            t1b = jnp.array(amplitudes["t1b"])
            t2aa = jnp.array(amplitudes["t2aa"])
            t2ab = jnp.array(amplitudes["t2ab"])
            t2bb = jnp.array(amplitudes["t2bb"])
            mo = [mo_coeff[0][:,:nocc_a], mo_coeff[1][:,:nocc_b]]
            mo_t = trial._thouless(mo, [t1a, t1b])
            wave_data['mo_ta'] = mo_t[0]
            wave_data['mo_tb'] = mo_t[1]
            wave_data["t2aa"] = t2aa
            wave_data["t2bb"] = t2bb
            wave_data["t2ab"] = t2ab
            wave_data['tau'] = trial.decompose_t2([t2aa,t2ab,t2bb])
            wave_data["mo_coeff"] = [mo_coeff[0][:, : nocc_a], mo_coeff[1][:, : nocc_b]]

            sampler = sampling.sampler_stoccsd2(
                n_prop_steps = options["n_prop_steps"],
                n_blocks = options["n_blocks"],
                n_chol = nchol,
                )
    

    if options["walker_type"] == "rhf":
        prop = propagation.propagator_restricted(
                options["dt"], 
                options["n_walkers"], 
                options["n_exp_terms"],
                options["n_batch"]
            )

    elif options["walker_type"] == "uhf":
        prop = propagation.propagator_unrestricted(
                options["dt"],
                options["n_walkers"],
                options["n_exp_terms"],
                options["n_batch"],
            )

    if  'pt' in options['trial'] and 'cc' in options['trial']:
        if 'pt2' in options['trial']:
            sampler = sampling.sampler_pt2(
                options["n_prop_steps"],
                options["n_blocks"],
                nchol,)
        else:
            sampler = sampling.sampler_pt(
                options["n_prop_steps"],
                options["n_blocks"],
                nchol,)
            
    elif 'stoccsd' in options['trial']:
        if '2' in options['trial']:
            sampler = sampling.sampler_stoccsd2(
                options["n_prop_steps"],
                options["n_blocks"],
                nchol,)
        else:
            sampler = sampling.sampler_stoccsd(
                options["n_prop_steps"],
                options["n_blocks"],
                nchol,)
            
    else:
        sampler = sampling.sampler(
            options["n_prop_steps"],
            options["n_blocks"],
            nchol,)

    
    if options["free_projection"]:
        if 'pt2' not in options["trial"]:
            sampler = fp_sampling.fp_sampler(
                    options["n_prop_steps"],
                    options["n_eql_blocks"],
                    options["n_trj"],
                    nchol,
                    )
        elif 'pt2' in options["trial"]:
            sampler = fp_sampling.fp_sampler_pt2(
                    options["n_prop_steps"],
                    options["n_eql_blocks"],
                    options["n_trj"],
                    nchol,
                    )

    print("\nQMC System")
    print(f"Number of electrons: {nelec_sp}")
    print(f"Spin Multiplicity:   {ms}")
    print(f"Number of orbitals:  {norb}")
    print(f"Number of Chol:      {nchol}")

    print("\nQMC Parameters")
    for op in options:
        if options[op] is not None:
            print(f"{str(op):<20s}: {str(options[op]):>20s}")

    return ham_data, ham, prop, trial, trial_ad, wave_data, sampler, options

In [59]:
import time

import numpy as np
from jax import random
from jax import numpy as jnp

from afqmc import config, prep, sampling

from functools import partial

print = partial(print, flush=True)
init_time = time.time()

# print("\nAFQMC Started")
prep.print_start()
config.setup_jax()

ham_data, ham, prop, trial, trial_ad, wave_data, sampler, options = (init_afqmc(options))

if "rdm1" not in wave_data:
    wave_data["rdm1"] = trial.get_rdm1(wave_data)
ham_data = ham.build_measurement_intermediates(ham_data, trial, wave_data)
ham_data = ham.build_propagation_intermediates(ham_data, prop, trial, wave_data)
h0 = ham_data['h0']

prop_data = prop.init_prop_data(trial, wave_data, ham_data, init_walkers=None)
if jnp.abs(jnp.sum(prop_data["overlaps"])) < 1.0e-6:
    raise ValueError(
        "Initial overlaps are zero. Pass walkers with non-zero overlap."
    )

prop_data["key"] = random.PRNGKey(options["seed"])
# prop_data["overlaps"] = trial.calc_overlap(prop_data["walkers"], wave_data)
prop_data["n_killed_walkers"] = 0

t1, t2, e0, e1 = trial.calc_energy_pt(prop_data["walkers"], ham_data, wave_data)
ept_sp = h0 + e0/t1 + e1/t1 - t2 * e0 / t1**2
ept = jnp.real(jnp.sum(ept_sp) / prop.n_walkers)

prop_data["pop_control_ene_shift"] = prop_data["e_estimate"]
print(f'initial AFQMC/pt2CCSD Energy is {ept:.6f}')

# prop_data = prep.init_ccsd_prop_data(
#     wave_data, ham_data, prop, trial,
#     options["n_walkers"], options["walker_type"], options["seed"],
# )

init_e = prop_data["e_estimate"]
init_w = np.sum(prop_data["weights"])


    ________                     _____                    
    ___  __ \___  __________________(_)_____________ _    
    __  /_/ /  / / /_  __ \_  __ \_  /__  __ \_  __ `/    
    _  _, _// /_/ /_  / / /  / / /  / _  / / /  /_/ /     
    /_/ |_| \__,_/ /_/ /_//_/ /_//_/  /_/ /_/_\__, /      
                                             /____/       
    _____________________________  ___________            
    ___    |__  ____/_  __ \__   |/  /_  ____/            
    __  /| |_  /_   _  / / /_  /|_/ /_  /                 
    _  ___ |  __/   / /_/ /_  /  / / / /___               
    /_/  |_/_/      \___\_\/_/  /_/  \____/               

Hostname:     sharmagroup-rn
System:       Linux
Node:         sharmagroup-rn
Release:      6.17.0-29-generic
Machine:      x86_64
Processor:    x86_64
JAX backend:  GPU
JAX devices:  [CudaDevice(id=0)]
Device kind:  NVIDIA GeForce RTX 5060 Ti
Platform:     gpu

Load system from Integral File
Maximum memory per walker:            2000.00 MB
Maximu

In [60]:
mycc.e_tot - ept

Array(-3.3443045e-06, dtype=float64)

In [35]:
print(abs(ham_data["chol"].max()))
print(abs(ham_data["h1"][0].max()))
print(abs(ham_data["h1"][1].max()))

0.6006734620078735
7.4584913645310165
7.455020548180854


In [116]:
# @partial(jit, static_argnums=0)
def _calc_energy_pt2_original(
    self,
    walker_up: jax.Array,
    walker_dn: jax.Array,
    ham_data: dict,
    wave_data: dict,
) -> complex:
    nocc_a, t2_aa = self.nelec[0], wave_data["t2aa"]
    nocc_b, t2_bb = self.nelec[1], wave_data["t2bb"]
    t2_ab = wave_data["t2ab"]
    mo_a, mo_b = wave_data['mo_ta'], wave_data['mo_tb']
    chol_a = ham_data["chol"][0].reshape(-1, self.norb, self.norb)
    chol_b = ham_data["chol"][1].reshape(-1, self.norb, self.norb)
    h1_a = ham_data["h1"][0]
    h1_b = ham_data["h1"][1]

    # full green's function G_pq
    green_a = (walker_up @ (jnp.linalg.inv(mo_a.T @ walker_up)) @ mo_a.T).T
    green_b = (walker_dn @ (jnp.linalg.inv(mo_b.T @ walker_dn)) @ mo_b.T).T
    greenp_a = (green_a - jnp.eye(self.norb))[:,nocc_a:]
    greenp_b = (green_b - jnp.eye(self.norb))[:,nocc_b:]

    hg_a = oe.contract("pq,pq->", h1_a, green_a, backend="jax")
    hg_b = oe.contract("pq,pq->", h1_b, green_b, backend="jax")
    hg = hg_a + hg_b # <exp(T1)HF|h1|walker>/<exp(T1)HF|walker>

    # <exp(T1)HF|h1|walker>/<exp(T1)HF|walker>
    # one body energy
    e1_0 = hg

    # <exp(T1)HF|T2 h1|walker>/<exp(T1)HF|walker>
    # double excitations
    t2g_a = oe.contract("iajb,ia->jb", t2_aa, green_a[:nocc_a,nocc_a:],
                        backend="jax") / 4
    t2g_b = oe.contract("iajb,ia->jb", t2_bb, green_b[:nocc_b,nocc_b:], 
                        backend="jax") / 4
    t2g_ab_a = oe.contract("iajb,jb->ia", t2_ab, green_b[:nocc_b,nocc_b:],
                            backend="jax")
    t2g_ab_b = oe.contract("iajb,ia->jb", t2_ab, green_a[:nocc_a,nocc_a:],
                            backend="jax")
    # t_iajb (G_ia G_jb - G_ib G_ja)
    gt2g_a = oe.contract("jb,jb->", t2g_a, green_a[:nocc_a,nocc_a:], 
                        backend="jax")
    gt2g_b = oe.contract("jb,jb->", t2g_b, green_b[:nocc_b,nocc_b:], 
                        backend="jax")
    gt2g_ab = oe.contract("ia,ia->", t2g_ab_a, green_a[:nocc_a,nocc_a:], 
                            backend="jax")
    gt2g = 2 * (gt2g_a + gt2g_b) + gt2g_ab # <exp(T1)HF|T2|walker>/<exp(T1)HF|walker>

    e1_2_1 = hg * gt2g
    
    t2_green_a = (greenp_a @ t2g_a.T) @ green_a[:nocc_a,:] # Gp_pb t_iajb G_ia G_jq
    t2_green_ab_a = (greenp_a @ t2g_ab_a.T) @ green_a[:nocc_a,:]
    t2_green_b = (greenp_b @ t2g_b.T) @ green_b[:nocc_b,:]
    t2_green_ab_b = (greenp_b @ t2g_ab_b.T) @ green_b[:nocc_b,:]
    e1_2_2_a = -oe.contract(
        "pq,pq->", h1_a, 4 * t2_green_a + t2_green_ab_a, backend="jax")
    e1_2_2_b = -oe.contract(
        "pq,pq->", h1_b, 4 * t2_green_b + t2_green_ab_b, backend="jax")
    e1_2_2 = e1_2_2_a + e1_2_2_b
    e1_2 = e1_2_1 + e1_2_2  # <exp(T1)HF|T2 h1|walker>/<exp(T1)HF|walker>

    # <exp(T1)HF|T2 h2|walker>/<exp(T1)HF|walker>
    # double excitations
    # e2_2_1 = e2_0 * gt2g
    lg_a = oe.contract("gpq,pq->g", chol_a, green_a, backend="jax")
    lg_b = oe.contract("gpq,pq->g", chol_b, green_b, backend="jax")
    lt2g_a = oe.contract("gpq,pq->g",
                        chol_a, 8 * t2_green_a + 2 * t2_green_ab_a,
                        backend="jax")
    lt2g_b = oe.contract(
        "gpq,pq->g", 
        chol_b, 
        8 * t2_green_b + 2 * t2_green_ab_b,
        backend="jax")
    e2_2_2_1 = -((lt2g_a + lt2g_b) @ (lg_a + lg_b)) / 2.0

    nchol = chol_a.shape[0]
    nchol_chunk = self.nchol_chunk
    nchunks = -(-nchol // nchol_chunk)
    pad = nchunks * nchol_chunk - nchol

    chol_a = jnp.pad(chol_a, ((0, pad), (0, 0), (0, 0)))
    chol_b = jnp.pad(chol_b, ((0, pad), (0, 0), (0, 0)))
    chol_a_chunks = chol_a.reshape(nchunks, nchol_chunk, self.norb, self.norb)
    chol_b_chunks = chol_b.reshape(nchunks, nchol_chunk, self.norb, self.norb)

    def scanned_fun(carry, x):
        chol_a_c, chol_b_c = x  # each shape (nchol_chunk, norb, norb)

        # e2_0
        lg_a_c = oe.contract("gpr,qr->gpq", chol_a_c, green_a, backend="jax")
        lg_b_c = oe.contract("gpr,qr->gpq", chol_b_c, green_b, backend="jax")
        tr_a = oe.contract("gpp->g", lg_a_c, backend="jax")
        tr_b = oe.contract("gpp->g", lg_b_c, backend="jax")
        e2_0_1_c = jnp.sum((tr_a + tr_b) ** 2) / 2.0
        e2_0_2_c = -(
            oe.contract("gpq,gqp->", lg_a_c, lg_a_c, backend="jax")
            + oe.contract("gpq,gqp->", lg_b_c, lg_b_c, backend="jax")
        ) / 2.0
        carry[0] += e2_0_1_c + e2_0_2_c

        # e2_2
        gl_a_c = oe.contract("pr,grq->gpq", green_a, chol_a_c, backend="jax")
        gl_b_c = oe.contract("pr,grq->gpq", green_b, chol_b_c, backend="jax")
        lt2_green_a_c = oe.contract(
            "gpr,qr->gpq", chol_a_c, 8 * t2_green_a + 2 * t2_green_ab_a, backend="jax"
        )
        lt2_green_b_c = oe.contract(
            "gpr,qr->gpq", chol_b_c, 8 * t2_green_b + 2 * t2_green_ab_b, backend="jax"
        )
        carry[1] += (
            oe.contract("gpq,gpq->", gl_a_c, lt2_green_a_c, backend="jax")
            + oe.contract("gpq,gpq->", gl_b_c, lt2_green_b_c, backend="jax")
        ) / 2

        glgp_a_c = oe.contract(
            "giq,qa->gia", gl_a_c[:, :nocc_a, :], greenp_a, backend="jax"
        )
        glgp_b_c = oe.contract(
            "giq,qa->gia", gl_b_c[:, :nocc_b, :], greenp_b, backend="jax"
        )

        if self.mix_precision:
            glgp_a_c_mp = glgp_a_c.astype(jnp.complex64)
            glgp_b_c_mp = glgp_b_c.astype(jnp.complex64)
            t2_aa_mp = t2_aa.astype(jnp.float32)
            t2_ab_mp = t2_ab.astype(jnp.float32)
            t2_bb_mp = t2_bb.astype(jnp.float32)
        else:                
            glgp_a_c_mp = glgp_a_c
            glgp_b_c_mp = glgp_b_c
            t2_aa_mp = t2_aa
            t2_ab_mp = t2_ab
            t2_bb_mp = t2_bb

        l2t2_a = 0.5 * oe.contract(
            "gia,gjb,iajb->", glgp_a_c_mp, glgp_a_c_mp, t2_aa_mp, backend="jax"
        )
        l2t2_b = 0.5 * oe.contract(
            "gia,gjb,iajb->", glgp_b_c_mp, glgp_b_c_mp, t2_bb_mp, backend="jax"
        )
        l2t2_ab = oe.contract(
            "gia,gjb,iajb->", glgp_a_c_mp, glgp_b_c_mp, t2_ab_mp, backend="jax"
        )
        carry[2] += (l2t2_a + l2t2_b + l2t2_ab).astype(jnp.complex128)
        return carry, 0.0

    [e2_0, e2_2_2_2, e2_2_3], _ = lax.scan(
        scanned_fun, [0.0, 0.0, 0.0], (chol_a_chunks, chol_b_chunks)
    )

    e2_2_1 = e2_0 * gt2g
    e2_2_2 = e2_2_2_1 + e2_2_2_2
    e2_2 = e2_2_1 + e2_2_2 + e2_2_3 # <exp(T1)HF|T2 h2|walker>/<exp(T1)HF|walker>

    o0 = jnp.linalg.det(walker_up[:nocc_a,:nocc_a]
        ) * jnp.linalg.det(walker_dn[:nocc_b,:nocc_b])
    # <exp(T1)HF|walker>/<HF|walker>
    t1 = jnp.linalg.det(wave_data["mo_ta"].T.conj() @ walker_up
        ) * jnp.linalg.det(wave_data["mo_tb"].T.conj() @ walker_dn) / o0
    t2 = gt2g * t1 # <exp(T1)HF|T2|walker>/<HF|walker>
    e0 = (e1_0 + e2_0) * t1 # <exp(T1)HF|h1+h2|walker>/<HF|walker>
    e1 = (e1_2 + e2_2) * t1 # <exp(T1)HF|T2 (h1+h2)|walker>/<HF|walker>

    return t1, t2, e0, e1

In [150]:
# @partial(jit, static_argnums=0)
def _calc_energy_pt2_mix_precision(
    self,
    walker_up: jax.Array,
    walker_dn: jax.Array,
    ham_data: dict,
    wave_data: dict,
) -> complex:
    # # only do this for two-body energy
    if self.mix_precision:
        rtype = jnp.float32
        ctype = jnp.complex64
    else:
        rtype = jnp.float64
        ctype = jnp.complex128

    nocc_a, t2_aa = self.nelec[0], wave_data["t2aa"]
    nocc_b, t2_bb = self.nelec[1], wave_data["t2bb"]
    t2_ab = wave_data["t2ab"]
    mo_a, mo_b = wave_data['mo_ta'], wave_data['mo_tb']
    chol_a = ham_data["chol"][0].reshape(-1, self.norb, self.norb)
    chol_b = ham_data["chol"][1].reshape(-1, self.norb, self.norb)
    h1_a = ham_data["h1"][0]
    h1_b = ham_data["h1"][1]

    # full green's function G_pq
    green_a = (walker_up @ (jnp.linalg.inv(mo_a.T @ walker_up)) @ mo_a.T).T
    green_b = (walker_dn @ (jnp.linalg.inv(mo_b.T @ walker_dn)) @ mo_b.T).T
    greenp_a = (green_a - jnp.eye(self.norb))[:,nocc_a:]
    greenp_b = (green_b - jnp.eye(self.norb))[:,nocc_b:]

    hg_a = oe.contract("pq,pq->", h1_a, green_a, backend="jax")
    hg_b = oe.contract("pq,pq->", h1_b, green_b, backend="jax")
    hg = hg_a + hg_b # <exp(T1)HF|h1|walker>/<exp(T1)HF|walker>

    # <exp(T1)HF|h1|walker>/<exp(T1)HF|walker>
    # one body energy
    e1_0 = hg

    # <exp(T1)HF|T2 h1|walker>/<exp(T1)HF|walker>
    # double excitations
    t2g_a = oe.contract("iajb,ia->jb", t2_aa, green_a[:nocc_a,nocc_a:], backend="jax") / 4
    t2g_b = oe.contract("iajb,ia->jb", t2_bb, green_b[:nocc_b,nocc_b:], backend="jax") / 4
    t2g_ab_a = oe.contract("iajb,jb->ia", t2_ab, green_b[:nocc_b,nocc_b:], backend="jax")
    t2g_ab_b = oe.contract("iajb,ia->jb", t2_ab, green_a[:nocc_a,nocc_a:], backend="jax")
    # t_iajb (G_ia G_jb - G_ib G_ja)
    gt2g_a = oe.contract("jb,jb->", t2g_a, green_a[:nocc_a,nocc_a:], backend="jax")
    gt2g_b = oe.contract("jb,jb->", t2g_b, green_b[:nocc_b,nocc_b:], backend="jax")
    gt2g_ab = oe.contract("ia,ia->", t2g_ab_a, green_a[:nocc_a,nocc_a:], backend="jax")
    gt2g = 2 * (gt2g_a + gt2g_b) + gt2g_ab # <exp(T1)HF|T2|walker>/<exp(T1)HF|walker>
    e1_2_1 = hg * gt2g
    
    t2_green_aaa = (greenp_a @ t2g_a.T) @ green_a[:nocc_a,:] # Gp_pb t_iajb G_ia G_jq
    t2_green_aba = (greenp_a @ t2g_ab_a.T) @ green_a[:nocc_a,:]
    t2_green_bbb = (greenp_b @ t2g_b.T) @ green_b[:nocc_b,:]
    t2_green_abb = (greenp_b @ t2g_ab_b.T) @ green_b[:nocc_b,:]
    t2_green_a_a = 4 * t2_green_aaa + t2_green_aba # connect a->a
    t2_green_b_b = 4 * t2_green_bbb + t2_green_abb # connect b->b

    e1_2_2_a = -oe.contract("pq,pq->", h1_a, t2_green_a_a, backend="jax")
    e1_2_2_b = -oe.contract("pq,pq->", h1_b, t2_green_b_b, backend="jax")
    e1_2_2 = e1_2_2_a + e1_2_2_b
    e1_2 = e1_2_1 + e1_2_2  # <exp(T1)HF|T2 h1|walker>/<exp(T1)HF|walker>

    # <exp(T1)HF|T2 h2|walker>/<exp(T1)HF|walker>
    # double excitations
    nchol = chol_a.shape[0]
    nchol_chunk = self.nchol_chunk
    nchunks = -(-nchol // nchol_chunk)
    pad = nchunks * nchol_chunk - nchol

    chol_a = jnp.pad(chol_a, ((0, pad), (0, 0), (0, 0)))
    chol_b = jnp.pad(chol_b, ((0, pad), (0, 0), (0, 0)))
    chol_a_chunks = chol_a.reshape(nchunks, nchol_chunk, self.norb, self.norb)
    chol_b_chunks = chol_b.reshape(nchunks, nchol_chunk, self.norb, self.norb)

    def scanned_fun(carry, x):
        chol_a_c, chol_b_c = x  # each shape (nchol_chunk, norb, norb)

        # e2_0
        gl_a_c = oe.contract("pr,gqr->gpq",
                             green_a.astype(jnp.complex128),
                             chol_a_c.astype(jnp.float64), 
                             backend="jax").astype(jnp.complex128)
        gl_b_c = oe.contract("pr,gqr->gpq", 
                             green_b.astype(jnp.complex128),
                             chol_b_c.astype(jnp.float64), 
                             backend="jax")
        tr_gl_a = oe.contract("gpp->g", 
                              gl_a_c.astype(jnp.complex128), 
                              backend="jax").astype(jnp.complex128)
        tr_gl_b = oe.contract("gpp->g", 
                              gl_b_c.astype(jnp.complex128), 
                              backend="jax").astype(jnp.complex128)
        ex_gl_a = oe.contract("gpq,gqp->g", 
                              gl_a_c.astype(jnp.complex128), 
                              gl_a_c.astype(jnp.complex128), 
                              backend="jax").astype(jnp.complex128)
        ex_gl_b = oe.contract("gpq,gqp->g", 
                              gl_b_c.astype(jnp.complex128), 
                              gl_b_c.astype(jnp.complex128), 
                              backend="jax").astype(jnp.complex128)
        e2_0_1_c = jnp.sum((tr_gl_a + tr_gl_b) ** 2) / 2.0
        e2_0_2_c = -jnp.sum(ex_gl_a + ex_gl_b) / 2.0

        carry[0] += (e2_0_1_c + e2_0_2_c).astype(jnp.complex128)

        # e2_2
        lt2g_a_c = oe.contract("gpr,qr->gpq", 
                               chol_a_c.astype(rtype), 
                               (2*t2_green_a_a).astype(ctype), 
                               backend="jax")
        lt2g_b_c = oe.contract("gpr,qr->gpq", 
                               chol_b_c.astype(rtype), 
                               (2*t2_green_b_b).astype(ctype), 
                               backend="jax")
        tr_lt2g_a_c = oe.contract("gqq->g", lt2g_a_c.astype(ctype), backend="jax")
        tr_lt2g_b_c = oe.contract("gqq->g", lt2g_b_c.astype(ctype), backend="jax")
        carry[1] += -(((tr_lt2g_a_c + tr_lt2g_b_c) @ (tr_gl_a + tr_gl_b)) / 2.0).astype(jnp.complex128)
        carry[2] += ((oe.contract("gpq,gpq->", gl_a_c, lt2g_a_c, backend="jax")
                    + oe.contract("gpq,gpq->", gl_b_c, lt2g_b_c, backend="jax")) 
                    / 2).astype(jnp.complex128)

        glgp_a_c = oe.contract("giq,qa->gia",
                               gl_a_c[:,:nocc_a,:].astype(ctype), 
                               greenp_a.astype(ctype), 
                               backend="jax")
        glgp_b_c = oe.contract("giq,qa->gia", 
                               gl_b_c[:,:nocc_b,:].astype(ctype), 
                               greenp_b.astype(ctype), 
                               backend="jax")

        l2t2_a = 0.5 * oe.contract("gia,gjb,iajb->", 
                                   glgp_a_c.astype(ctype), 
                                   glgp_a_c.astype(ctype), 
                                   t2_aa.astype(rtype), 
                                   backend="jax")
        l2t2_b = 0.5 * oe.contract("gia,gjb,iajb->", 
                                   glgp_b_c.astype(ctype), 
                                   glgp_b_c.astype(ctype), 
                                   t2_bb.astype(rtype), 
                                   backend="jax")
        l2t2_ab = oe.contract("gia,gjb,iajb->", 
                              glgp_a_c.astype(ctype), 
                              glgp_b_c.astype(ctype), 
                              t2_ab.astype(rtype), 
                              backend="jax")
        carry[3] += (l2t2_a + l2t2_b + l2t2_ab).astype(jnp.complex128)
        return carry, 0.0

    [e2_0, e2_2_2_1, e2_2_2_2, e2_2_3], _ = lax.scan(
        scanned_fun, [0.0, 0.0, 0.0, 0.0], (chol_a_chunks, chol_b_chunks)
    )

    e2_2_1 = e2_0 * gt2g
    e2_2_2 = e2_2_2_1 + e2_2_2_2
    e2_2 = e2_2_1 + e2_2_2 + e2_2_3 # <exp(T1)HF|T2 h2|walker>/<exp(T1)HF|walker>

    o0 = jnp.linalg.det(walker_up[:nocc_a,:nocc_a]
        ) * jnp.linalg.det(walker_dn[:nocc_b,:nocc_b])
    # <exp(T1)HF|walker>/<HF|walker>
    t1 = jnp.linalg.det(wave_data["mo_ta"].T.conj() @ walker_up
        ) * jnp.linalg.det(wave_data["mo_tb"].T.conj() @ walker_dn) / o0
    t2 = gt2g * t1 # <exp(T1)HF|T2|walker>/<HF|walker>
    e0 = (e1_0 + e2_0) * t1 # <exp(T1)HF|h1+h2|walker>/<HF|walker>
    e1 = (e1_2 + e2_2) * t1 # <exp(T1)HF|T2 (h1+h2)|walker>/<HF|walker>

    return t1, t2, e0, e1

In [107]:
prop_data["walkers"][0][0].shape
prop_data["walkers"][1][0].shape

(58, 4)

In [155]:
trial.mix_precision = True
print(trial.mix_precision)

True


In [152]:
walker_up = jnp.array(np.random.rand(*prop_data["walkers"][0][0].shape) \
                      + 1j *np.random.rand(*prop_data["walkers"][0][0].shape)) / 2
walker_dn = jnp.array(np.random.rand(*prop_data["walkers"][1][0].shape) \
                      + 1j *np.random.rand(*prop_data["walkers"][1][0].shape)) / 2

In [156]:
t1m, t2m, e0m, e1m = _calc_energy_pt2_mix_precision(trial, walker_up, walker_dn, ham_data, wave_data)
ept2m = h0 + e0m/t1m + e1m/t1m - t2m*e0m/t1m**2
print(t1m, t2m, e0m, e1m, ept2m)
t1o, t2o, e0o, e1o = _calc_energy_pt2_original(trial, walker_up, walker_dn, ham_data, wave_data)
ept2o = h0 + e0o/t1o + e1o/t1o - t2o*e0o/t1o**2
print(t1o, t2o, e0o, e1o, ept2o)

(0.9593805352723999-0.07711809045614859j) (-0.006664039766621876-0.05703739366831024j) (-28.311171726439632+1.7132374477867969j) (-0.5435250655465296+2.3088846607153983j) (-108.74169410451493+0.00538659784394091j)
(0.9593805352723999-0.07711809045614859j) (-0.006664039766621876-0.05703739366831024j) (-28.311171726439635+1.7132374477867924j) (-0.5437475814353776+2.3087066737166944j) (-108.74190973535043+0.005183741907396122j)


In [154]:
t1a, t2a, e0a, e1a = trial_ad._calc_energy_pt(walker_up, walker_dn, ham_data, wave_data)
ept2a = h0 + e0a/t1a + e1a/t1a - t2a*e0a/t1a**2
print(t1a, t2a, e0a, e1a, ept2a)

(0.9593805352723997-0.07711809045615707j) (-0.00666403976662294-0.05703739366831032j) (-28.311171726439575+1.7132374477870562j) (-0.543556620890256+2.308753563618101j) (-108.74171587106754+0.005248200526730473j)
